In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

invoice_img = Image.open(
    "../../data/gemini_generated_invoice.png"
).convert("RGB")


width, height = invoice_img.size   # Get dimensions
left = width/3.66
top = height/4
right = 3 * width/4
bottom = 4 * height/4
invoice_img = invoice_img.crop((left, top, right, bottom))


new_size = (invoice_img.width // 2, invoice_img.height // 2)
invoice_img = invoice_img.resize(new_size, Image.LANCZOS)

invoice_img

In [ ]:
def extract_patches(img_np, patch_size=16):
    """
    Args:
        img_np (np.ndarray): Image array of shape (H, W, C)
    Returns:
        list of np.ndarray: List of flat or 3D patches
    """
    H, W, C = img_np.shape
    num_patches_h = H // patch_size
    num_patches_w = W // patch_size
    
    patches = []
    for i in range(num_patches_h):
        for j in range(num_patches_w):
            # Calculate pixel boundaries
            start_h = i * patch_size
            start_w = j * patch_size
            
            # --- TODO 1 ---
            # Extract the correct 16x16xC slice from img_np using the start boundaries
            patch = img_np[______ : ______, start_w : start_w + ______, :] # TODO: Fill in slicing boundaries
            
            patches.append(patch)
    return patches, num_patches_h, num_patches_w

# Convert PIL to Numpy and patchify
img_np = np.array(invoice_img)

patches, n_h, n_w = extract_patches(img_np, patch_size=16)

print(f"Total patches generated: {len(patches)} (Expected: 14x14 = 196 patches)")

# Visualize the first 16 patches
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for idx, ax in enumerate(axes.flat):
    ax.imshow(patches[idx])
    ax.axis('off')
plt.suptitle("First 16 Visual Word Patches")
plt.show()

In [ ]:
class ViTPatchEmbedding(nn.Module):
    def __init__(self, patch_size=16, in_channels=3, embed_dim=128):
        super().__init__()
        self.patch_size = patch_size
        # A 16x16x3 patch flattens into a 768-dimensional vector
        self.flat_dim = patch_size * patch_size * in_channels
        
        # --- TODO 2 ---
        # Define a linear layer that projects the flat patch vector to the hidden 'embed_dim'
        self.projection = nn.Linear(in_features=_______, out_features=_______) 

    def forward(self, patches_list):
        """
        Args:
            patches_list (list): List of N numpy array patches
        Returns:
            torch.Tensor: Shape (1, N, embed_dim) -> Batch size 1 for simplicity
        """
        # Flatten each patch from (16, 16, 3) -> (768,)
        flat_patches = [p.flatten() for p in patches_list]
        
        # Convert list to a PyTorch tensor and add batch dimension -> (1, N, 768)
        x = torch.tensor(np.array(flat_patches), dtype=torch.float32).unsqueeze(0)
        
        # Project tokens into embedding space
        projected_embeddings = self.projection(x)
        return projected_embeddings

# Test the projection
patch_embedder = ViTPatchEmbedding(patch_size=16, in_channels=3, embed_dim=128)
vision_tokens = patch_embedder(patches)
print("Vision Tokens Tensor Shape:", vision_tokens.shape) 
# Expected Output: torch.Size([1, 196, 128])

## Questions
- How many images patches are there?
- What is the embedding dimension of each patch?

In [ ]:
class SimpleTextEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        # A tiny vocabulary dictionary for our specific use case
        self.vocab = {"<pad>": 0, "what": 1, "is": 2, "the": 3, "invoice": 4, "total?": 5, "1888,96$": 6}
        self.vocab_size = len(self.vocab)
        
        # --- TODO 3 ---
        # Text embedding matrix mapping word indices to embed_dim
        self.token_embeddings = nn.Embedding(num_embeddings=_______, embedding_dim=embed_dim)
        
    def tokenize_and_embed(self, text_query):
        words = text_query.lower().split()
        # Convert words to numerical IDs based on our vocabulary dictionary
        token_ids = [self.vocab.get(w, 0) for w in words] 
        
        # Convert to tensor and add batch dimension -> (1, sequence_length)
        token_tensor = torch.tensor([token_ids], dtype=torch.long)
        
        # Pass through embedding layer -> (1, sequence_length, embed_dim)
        text_embeddings = self.token_embeddings(token_tensor)
        return text_embeddings

text_encoder = SimpleTextEncoder(embed_dim=128)
text_tokens = text_encoder.tokenize_and_embed("what is the invoice total?")
print("Text Tokens Tensor Shape:", text_tokens.shape)

In [ ]:
class SimpleVLM(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.patch_embedder = ViTPatchEmbedding(patch_size=16, in_channels=3, embed_dim=embed_dim)
        self.text_encoder = SimpleTextEncoder(embed_dim=embed_dim)
        
        # Multimodal Fusion Transformer Block
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # Final classification head predicting the answer token (e.g., "$120.00")
        self.answer_head = nn.Linear(embed_dim, 1) 

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=8,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.lm_head = nn.Linear(embed_dim, self.text_encoder.vocab_size)

    def forward(self, raw_patches, text_query, target_txt):
        # 1. Extract Embeddings
        vis_emb = self.patch_embedder(raw_patches)  # Shape: (1, 196, 128)
        text_emb = self.text_encoder.tokenize_and_embed(text_query)  # Shape: (1, 5, 128)
        
        # --- TODO 4 ---
        # Concatenate vision embeddings and text embeddings along the sequence length axis (dim 1)
        # Hint: Final sequence length should be 196 + 5 = 201 tokens
        multimodal_tokens = torch.cat((_______, _______), dim=_______) 
        
        # 2. Process via Fusion Transformer
        fused_outputs = self.fusion_transformer(multimodal_tokens) # Shape: (1, 201, 128)
        
        # 3. Predict answer from the contextualized sequence
        # causal mask
        txt = self.text_encoder.tokenize_and_embed(target_txt)
        T = txt.size(1)
        causal_mask = torch.triu(
            torch.full((T, T), float("-inf"), device=txt.device),
            diagonal=1
        )

        # decoder cross-attends to image tokens
        out = self.decoder(
            tgt=txt,
            memory=fused_outputs,
            tgt_mask=causal_mask
        )

        return self.lm_head(out)


# Instantiate the full VLM
vlm = SimpleVLM(embed_dim=128)

# Execute Model Forward Pass
question = "what is the invoice total?"
answer = "1888,96$"
output_logits = vlm(patches, question, answer)

print("\n--- Evaluation Check ---")
print("Combined Multimodal Sequence Shape processed successfully!")
print("VLM Output logit shape:", output_logits.shape)